# NeuralGCM 1-year climate simulation — St. John's, NL

Run a free-running NeuralGCM forecast for one full calendar year at 2.8° resolution, with time-varying SST + sea-ice forcings from ERA5, and compare the simulated trajectory at the St. John's grid point against ERA5 reanalysis on a single chart.

**Do not run this on a local laptop.** A one-year unroll takes hours on CPU. Use a cluster, Colab, or a workstation with a GPU/TPU.

In [2]:
! pip install -q -U neuralgcm dinosaur gcsfs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.0/175.0 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.7/77.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.6/202.6 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.3/374.3 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 307.5/307.5 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 94.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.3.0 which is incompatible.

In [ ]:
import jax
import numpy as np
import matplotlib.pyplot as plt

import gcsfs  # Google Cloud Storage filesystem - reading directly from GCS storage buckets
# Python's serialization module - NeuralGCM model checkpoints (weights + configs) are stored as pickle files
import pickle
# N-dimensional labeled arrary library - ERA5 data comes in this format (dimensions: time, level, latitude, longitude)
import xarray

# dinosaur is neuralgcm's dynamics package
from dinosaur import horizontal_interpolation
from dinosaur import spherical_harmonic
from dinosaur import xarray_utils

import neuralgcm

### Load a pre-trained NeuralGCM model


In [ ]:
# @param ['v1/deterministic_0_7_deg.pkl', 'v1/deterministic_1_4_deg.pkl', 'v1/deterministic_2_8_deg.pkl', 'v1/stochastic_1_4_deg.pkl', 'v1_precip/stochastic_precip_2_8_deg.pkl', 'v1_precip/stochastic_evap_2_8_deg.pkl'] {type: "string"}
model_name = 'v1/deterministic_2_8_deg.pkl'

gcs = gcsfs.GCSFileSystem(token='anon')
with gcs.open(f'gs://neuralgcm/models/{model_name}', 'rb') as f:
    ckpt = pickle.load(f)

model = neuralgcm.PressureLevelModel.from_checkpoint(ckpt)

### Load ERA5 data from GCP/Zarr

Because we are loading the data from Zarr / GCP (Google Cloud Platform), we don't need to download the full 30 years dataset.

In [ ]:
# Open the ERA5 zarr store lazily (no data is downloaded yet).
# The actual one-year slice is selected later in the climate-sim section.
era5_path = 'gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3'
full_era5 = xarray.open_zarr(
    era5_path, chunks=None, storage_options=dict(token='anon')
)

Regrid to NeuralGCM’s native resolution:

In [ ]:
# Build a conservative regridder from ERA5's 0.25° grid to NeuralGCM's native 2.8° Gauss grid.
era5_grid = spherical_harmonic.Grid(
    latitude_nodes=full_era5.sizes['latitude'],
    longitude_nodes=full_era5.sizes['longitude'],
    latitude_spacing=xarray_utils.infer_latitude_spacing(full_era5.latitude),
    longitude_offset=xarray_utils.infer_longitude_offset(full_era5.longitude),
)
regridder = horizontal_interpolation.ConservativeRegridder(
    era5_grid, model.data_coords.horizontal, skipna=True
)

Let's check the resolution of the `full_era5` dataset.

In [13]:
print(f"ERA5 Latitude: {full_era5.latitude.min().item()} to {full_era5.latitude.max().item()} with {len(full_era5.latitude)} points.")
print(f"ERA5 Longitude: {full_era5.longitude.min().item()} to {full_era5.longitude.max().item()} with {len(full_era5.longitude)} points.")

# Calculate approximate resolution
lat_diff = np.abs(full_era5.latitude.diff('latitude').item(0))
lon_diff = np.abs(full_era5.longitude.diff('longitude').item(0))
print(f"Approximate Latitude Resolution: {lat_diff:.2f} degrees")
print(f"Approximate Longitude Resolution: {lon_diff:.2f} degrees")

ERA5 Latitude: -90.0 to 90.0 with 721 points.
ERA5 Longitude: 0.0 to 359.75 with 1440 points.
Approximate Latitude Resolution: 0.25 degrees
Approximate Longitude Resolution: 0.25 degrees


The above results show that, ERA5 has an approximate resolution of about 0.25degs

## One-year free-running simulation vs ERA5 at St. John's

Run NeuralGCM 2.8° for one full year from a single Jan-1 initial condition, with real time-varying SST + sea-ice forcings from ERA5. Then extract the St. John's grid point and compare against ERA5 reanalysis.

**Important caveats — read before interpreting the chart**

- NeuralGCM here is a **free-running** forecast (no data assimilation). After ~2 weeks the simulated weather decorrelates from the real atmosphere — so day-to-day values *should not* match ERA5 past the first fortnight.
- What is meaningful at this scale: **monthly means, annual cycle, variance, distribution** at the location. Those should track ERA5 closely if the model is well-tuned.
- Runtime: hours on CPU; minutes on GPU. Don't run this locally on the Mac — push to the cluster.
- Output is the full 2.8° global field at daily cadence (~2–3 GB in memory). Reduce variables or thin time if memory is tight.

In [ ]:
# --- Config ---
sim_year = 2020
sim_start = f'{sim_year}-01-01'
sim_end   = f'{sim_year}-12-31'
data_inner_steps_yr = 24   # take ERA5 every 24h (daily)

# St. John's, NL — used for grid-point extraction below
stjohns_lat = 47.5615
stjohns_lon = 360.0 - 52.7126   # ≈ 307.29 °E (0–360 convention)

# --- Load + regrid one year of ERA5 (reuse the existing regridder) ---
sliced_era5_yr = (
    full_era5
    [model.input_variables + model.forcing_variables]
    .pipe(
        xarray_utils.selective_temporal_shift,
        variables=model.forcing_variables,
        time_shift='24 hours',
    )
    .sel(time=slice(sim_start, sim_end, data_inner_steps_yr))
    .compute()
)
eval_era5_yr = xarray_utils.regrid(sliced_era5_yr, regridder)
eval_era5_yr = xarray_utils.fill_nan_with_nearest(eval_era5_yr)

print(f"Loaded {eval_era5_yr.sizes['time']} daily ERA5 snapshots "
      f"for {sim_year} on the {model.data_coords.horizontal.latitude_nodes}×"
      f"{model.data_coords.horizontal.longitude_nodes} model grid.")

In [ ]:
# --- 1-year unroll ---
inner_steps_yr = 24                       # save model output once per day
n_snapshots = eval_era5_yr.sizes['time']
outer_steps_yr = n_snapshots              # produce as many outputs as we have ERA5 days
timedelta_yr = np.timedelta64(1, 'h') * inner_steps_yr
times_yr = np.arange(outer_steps_yr) * inner_steps_yr

# Initial state from Jan 1
inputs_yr = model.inputs_from_xarray(eval_era5_yr.isel(time=0))
input_forcings_yr = model.forcings_from_xarray(eval_era5_yr.isel(time=0))
initial_state_yr = model.encode(inputs_yr, input_forcings_yr, jax.random.key(42))

# Real time-varying SST / sea-ice forcings over the whole year
all_forcings_yr = model.forcings_from_xarray(eval_era5_yr)

print(f"Unrolling {outer_steps_yr} daily steps "
      f"({outer_steps_yr/365:.2f} years)…")
final_state_yr, predictions_yr = model.unroll(
    initial_state_yr,
    all_forcings_yr,
    steps=outer_steps_yr,
    timedelta=timedelta_yr,
    start_with_input=True,
)
predictions_yr_ds = model.data_to_xarray(predictions_yr, times=times_yr)
print("Done.")

In [ ]:
# --- Build a combined dataset with real datetimes, then extract St. John's ---
target_yr = model.inputs_from_xarray(eval_era5_yr.isel(time=slice(outer_steps_yr)))
target_yr_ds = model.data_to_xarray(target_yr, times=times_yr)

combined_yr_ds = xarray.concat([target_yr_ds, predictions_yr_ds], 'model')
combined_yr_ds = combined_yr_ds.assign_coords(model=['ERA5', 'NeuralGCM'])

# Replace the integer-hour time axis with real datetimes
dates_yr = eval_era5_yr.time.isel(time=slice(outer_steps_yr)).values
combined_yr_ds = combined_yr_ds.assign_coords(time=dates_yr)

stjohns_yr = combined_yr_ds.sel(
    latitude=stjohns_lat, longitude=stjohns_lon, method='nearest'
)
print(f"Sampled grid point: lat={float(stjohns_yr.latitude):.2f}, "
      f"lon={float(stjohns_yr.longitude):.2f}")
stjohns_yr

In [ ]:
# --- Chart: daily timeseries (with 30-d rolling overlay) + monthly cycle + scatter ---
g = 9.80665
t_era  = stjohns_yr.temperature.sel(level=1000, model='ERA5').values     - 273.15
t_ngcm = stjohns_yr.temperature.sel(level=1000, model='NeuralGCM').values - 273.15
q_era  = stjohns_yr.specific_humidity.sel(level=850, model='ERA5').values     * 1000
q_ngcm = stjohns_yr.specific_humidity.sel(level=850, model='NeuralGCM').values * 1000
z_era  = stjohns_yr.geopotential.sel(level=500, model='ERA5').values     / g
z_ngcm = stjohns_yr.geopotential.sel(level=500, model='NeuralGCM').values / g

# 30-day centred rolling means for the top panel
t_era_30  = (stjohns_yr.temperature.sel(level=1000, model='ERA5')
             .rolling(time=30, center=True, min_periods=1).mean().values) - 273.15
t_ngcm_30 = (stjohns_yr.temperature.sel(level=1000, model='NeuralGCM')
             .rolling(time=30, center=True, min_periods=1).mean().values) - 273.15

monthly = stjohns_yr.groupby('time.month').mean('time')
t_era_m  = monthly.temperature.sel(level=1000, model='ERA5').values     - 273.15
t_ngcm_m = monthly.temperature.sel(level=1000, model='NeuralGCM').values - 273.15
months   = monthly.month.values

fig = plt.figure(figsize=(14, 8))
gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 1.2])

ax_t = fig.add_subplot(gs[0, :])
ax_t.plot(dates_yr, t_era,     color='C0', lw=0.7, alpha=0.35, label='ERA5 daily')
ax_t.plot(dates_yr, t_ngcm,    color='C1', lw=0.7, alpha=0.35, label='NeuralGCM daily')
ax_t.plot(dates_yr, t_era_30,  color='C0', lw=2.2,             label='ERA5 30-d mean')
ax_t.plot(dates_yr, t_ngcm_30, color='C1', lw=2.2,             label='NeuralGCM 30-d mean')
ax_t.set_ylabel('T 1000 hPa (°C)')
ax_t.legend(loc='upper right', ncol=2, fontsize=9)
ax_t.set_title(f"St. John's daily — {sim_year}  "
               f"(synoptic divergence after ~2 weeks is expected; "
               f"compare the 30-d traces)")
ax_t.grid(alpha=0.3)

ax_q = fig.add_subplot(gs[1, 0], sharex=ax_t)
ax_q.plot(dates_yr, q_era,  label='ERA5',      lw=1.0)
ax_q.plot(dates_yr, q_ngcm, label='NeuralGCM', lw=1.0, alpha=0.85)
ax_q.set_ylabel('q 850 hPa (g/kg)'); ax_q.legend(); ax_q.grid(alpha=0.3)

ax_z = fig.add_subplot(gs[1, 1], sharex=ax_t)
ax_z.plot(dates_yr, z_era,  label='ERA5',      lw=1.0)
ax_z.plot(dates_yr, z_ngcm, label='NeuralGCM', lw=1.0, alpha=0.85)
ax_z.set_ylabel('Z 500 hPa (m)'); ax_z.legend(); ax_z.grid(alpha=0.3)

ax_m = fig.add_subplot(gs[2, 0])
ax_m.plot(months, t_era_m,  marker='o', label='ERA5')
ax_m.plot(months, t_ngcm_m, marker='s', label='NeuralGCM')
ax_m.set(xlabel='month', ylabel='monthly mean T 1000 (°C)',
         title='Annual cycle', xticks=range(1, 13))
ax_m.legend(); ax_m.grid(alpha=0.3)

ax_s = fig.add_subplot(gs[2, 1])
ax_s.scatter(t_era, t_ngcm, s=8, alpha=0.4)
lo, hi = min(t_era.min(), t_ngcm.min()), max(t_era.max(), t_ngcm.max())
ax_s.plot([lo, hi], [lo, hi], 'k--', lw=1)
bias = float(np.mean(t_ngcm - t_era))
rmse = float(np.sqrt(np.mean((t_ngcm - t_era) ** 2)))
ax_s.set(xlabel='ERA5 T 1000 (°C)', ylabel='NeuralGCM T 1000 (°C)',
         title=f'Daily distribution\nbias={bias:+.2f} °C, RMSE={rmse:.2f} °C')
ax_s.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Monthly distribution of daily T1000 at St. John's

A scatter plot pairs daily values 1:1, which is misleading for a free-running model because the days are not expected to match. A box plot per month, in contrast, asks the right question: *for each month, do the daily values from NeuralGCM occupy the same range as ERA5?* That's the meaningful AMIP-style diagnostic.

In [ ]:
import matplotlib.patches as mpatches

t_era_da  = stjohns_yr.temperature.sel(level=1000, model='ERA5')     - 273.15
t_ngcm_da = stjohns_yr.temperature.sel(level=1000, model='NeuralGCM') - 273.15

era_by_month  = [t_era_da.where(t_era_da['time.month']  == m, drop=True).values
                 for m in range(1, 13)]
ngcm_by_month = [t_ngcm_da.where(t_ngcm_da['time.month'] == m, drop=True).values
                 for m in range(1, 13)]

fig, ax = plt.subplots(figsize=(12, 5))
positions = np.arange(1, 13)
width = 0.35

bp_e = ax.boxplot(era_by_month,  positions=positions - 0.2, widths=width,
                  patch_artist=True,
                  boxprops=dict(facecolor='#9ec9e8'),
                  medianprops=dict(color='#08306b'))
bp_n = ax.boxplot(ngcm_by_month, positions=positions + 0.2, widths=width,
                  patch_artist=True,
                  boxprops=dict(facecolor='#f4a582'),
                  medianprops=dict(color='#67000d'))

ax.set_xticks(positions)
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                    'Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_ylabel('Daily T 1000 hPa (°C)')
ax.set_title(f"St. John's daily T1000 distribution by month — {sim_year}")
ax.grid(alpha=0.3, axis='y')
ax.legend(handles=[mpatches.Patch(facecolor='#9ec9e8', label='ERA5'),
                   mpatches.Patch(facecolor='#f4a582', label='NeuralGCM')],
          loc='upper left')
plt.tight_layout()
plt.show()

### Annual-mean T1000 bias map around Newfoundland

Is the bias at St. John's a local artefact, or part of a broader regional pattern? Plot the annual-mean T1000 difference (NeuralGCM − ERA5) over Atlantic Canada. The star marks the city; the bias quoted in the box plot above is the value of the cell beneath it.

In [ ]:
# Annual-mean T1000 bias = NeuralGCM annual mean − ERA5 annual mean
annual_t = combined_yr_ds.temperature.sel(level=1000).mean('time')
bias_t = annual_t.sel(model='NeuralGCM') - annual_t.sel(model='ERA5')

# Region: Atlantic Canada + surrounding ocean
lat_min, lat_max = 35, 60
lon_min, lon_max = 280, 330
bias_region = bias_t.sel(
    latitude=slice(lat_min, lat_max),
    longitude=slice(lon_min, lon_max),
)

vmax = float(np.abs(bias_region).max())

fig, ax = plt.subplots(figsize=(9, 6))
bias_region.plot(
    x='longitude', y='latitude', ax=ax,
    cmap='RdBu_r', vmin=-vmax, vmax=vmax,
    cbar_kwargs={'label': 'Annual-mean T1000 bias (K)\nNeuralGCM − ERA5'},
)

ax.plot(stjohns_lon, stjohns_lat, marker='*', markersize=18,
        markerfacecolor='yellow', markeredgecolor='black',
        linestyle='None', label="St. John's")
ax.set_title(f"Annual-mean T1000 bias — {sim_year}")
ax.set_xlabel('Longitude (°E)'); ax.set_ylabel('Latitude (°N)')
ax.legend(loc='lower left')
plt.tight_layout()
plt.show()

# Bias at the sampled St. John's grid point
pt_bias = float(bias_t.sel(
    latitude=stjohns_lat, longitude=stjohns_lon, method='nearest'
))
print(f"Annual-mean bias at St. John's grid point: {pt_bias:+.2f} K")
print(f"Regional |bias| max in the plotted box: {vmax:.2f} K")